In [1]:
import pandas as pd
import numpy as np
from openpyxl import load_workbook
import os

In [ ]:
switch_page_status = {'Instructions': False, 'Video 1': False, 'Video 2': False,'03_q1a': True, '03_q1b': True, '03_q1c': False, '03_q2a': True,  '03_q2b': True, '03_q2c': False, '03_q3a': True, '03_q3b': True, '03_q3c': False, '03_q4a': True, '03_q4b': True, '03_q4c': False, '03_q5a': True, '03_q5b': True, '03_q5c': False, '04_q1a': True, '04_q1b': False, '04_q2a': True, '04_q2b': False, '04_q3a': True, '04_q3b': False, '04_q4a': True, '04_q4b': False, '04_q5a': True, '04_q5b': False}
black_screen_page = ['03_q1b', '03_q2a', '03_q2c', '03_q3a', '03_q5b', '04_q2a', '04_q2b', '04_q3a', '04_q4a', '04_q5a', '04_q5b']
def modify_label(wb, filename):
    ws = wb.active
    df = pd.read_excel(filename)
    df = df.replace({np.nan: None})
    labels = []
    status = False
    black_screen = False
    for index, row in df.iterrows():
        label = row['label']
        if row['action']:
            actions = row['action'].split(', ')           
            for action in actions:
                if action in switch_page_status:
                    status = switch_page_status[action]
                if action in black_screen_page:
                    black_screen = True
                if action == 'play_video':
                    black_screen = False
        if status and label:
            label = label.replace('quiz_attempt', '')
            label = label.replace('quiz_attempt, ', '')
            label = label.replace("content_reading", "quiz_attempt")
        if label:
            label = label.replace('irrelevant_reading', 'irrelevant_navigation')
            if black_screen:
                label = label.replace('content_reading', 'content_navigation')
                label = label.replace('irrelevant_navigation', 'content_navigation')
        labels.append(label)

    df['label'] = labels
    for row_idx, row in enumerate(df.values, start=2):
        for col_idx, value in enumerate(row, start=1):
            ws.cell(row=row_idx, column=col_idx, value=value)
    return df

In [5]:
for filename in os.listdir('data_video/labelled_VBL'):
    name = os.path.join('data_video/labelled_VBL', filename)
    wb  = load_workbook(name)
    df = modify_label(wb, name)
    new_path = 'data_video/modified_VBL/' + filename
    wb.save(new_path)
    